# Day 16 — Learning Rate & Training Optimization

## 1. Learning Objectives
- Understand why a static Learning Rate is often sub-optimal.
- Implement **StepLR** to drop the learning rate on a schedule.
- Implement **ReduceLROnPlateau** to drop the rate when validation loss stalls.
- Understand modern concepts like **Warmup** and **OneCycleLR**.

## 2. Prerequisites
- Optimizers (Day 11).
- Training Loops (Day 12).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler

## 3. Concept Explanation
The Learning Rate (`lr`) is the most important hyperparameter you will tune. 
- If `lr` is too high, the model overshoots the minimum and the loss explodes.
- If `lr` is too low, the model takes tiny steps and takes forever to train.

**Learning Rate Schedulers** solve this by dynamically changing the learning rate *during* training. Usually, we start with a large learning rate to learn fast, and then gradually decay it to take smaller, more precise steps as we approach the minimum.

## 4. PyTorch `lr_scheduler` API
Schedulers wrap around your optimizer. After you call `optimizer.step()`, you call `scheduler.step()`.

In [ ]:
# Setup a dummy model and optimizer
model = nn.Linear(10, 1)
optimizer = optim.Adam(model.parameters(), lr=0.1)

# 1. StepLR: Multiplies lr by gamma every step_size epochs.
# E.g., Every 10 epochs, multiply lr by 0.1
step_scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

In [ ]:
# Let's simulate 30 epochs of training and watch the learning rate change
lrs = []
for epoch in range(30):
    # ... forward, backward, optimizer.step() happen here ...
    optimizer.step() 
    
    # At the end of the epoch, step the scheduler
    step_scheduler.step()
    
    # Record the current LR (it's stored in optimizer.param_groups)
    current_lr = optimizer.param_groups[0]['lr']
    lrs.append(current_lr)
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch}, LR: {current_lr}")

## 5. ReduceLROnPlateau
Instead of guessing when to drop the learning rate, `ReduceLROnPlateau` watches your Validation Loss. If the loss stops improving for a certain number of epochs (the `patience`), it automatically drops the learning rate.

*Note: Because it needs to see the loss, you must pass the validation loss into `scheduler.step(val_loss)`.*

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.1)
plateau_scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# In a real loop:
# ... get val_loss ...
# plateau_scheduler.step(val_loss)

## 6. Advanced Concepts: Warmup & OneCycleLR
Modern architectures (like Transformers) are very unstable at the beginning of training. 
- **Warmup**: Start the learning rate at nearly 0, and linearly increase it to the target LR over the first few epochs. This stabilizes the early gradients.
- **OneCycleLR**: Increases the LR to a maximum, then anneals it down to near zero. It's heavily used in fast.ai and modern computer vision/NLP.

*Note: `OneCycleLR` steps every **batch**, not every epoch!*

## 11. Practice Exercise 1: Incorporating StepLR
Write out the 5 core training loop steps + the validation block. 
Where exactly should `scheduler.step()` be placed if you are using `StepLR`?

In [ ]:
# SOLUTION / PSEUDOCODE
# for epoch in range(epochs):
#     # --- Training ---
#     for batch in train_loader:
#         pred = model(x)
#         loss = criterion(pred, y)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#
#     # --- Validation ---
#     # ... calculate val loss ...
#
#     # --- Scheduler Step ---
#     scheduler.step() # Placed at the very end of the epoch!

## 13. Debugging Challenge
A developer is using `ReduceLROnPlateau`, but PyTorch throws a `TypeError: step() missing 1 required positional argument: 'metrics'`. Why?

**Solution:** Unlike `StepLR` which just steps based on the epoch count, `ReduceLROnPlateau` needs to know if the model is plateauing! You must pass the validation loss to it: `scheduler.step(val_loss)`.

## 17. Interview Questions
1. **Why do we decay the learning rate instead of keeping it constant?**
   *Answer*: A high learning rate allows the model to traverse the loss landscape quickly and escape local minima initially. However, as it gets close to the global minimum, a high learning rate will cause it to "bounce around" and never settle into the lowest point. Decaying the learning rate allows for fine-tuning.
2. **Why do Transformers typically require a learning rate Warmup?**
   *Answer*: At initialization, the network weights are completely random, causing extreme gradient variances. A large initial learning rate can cause the weights to explode or collapse before they learn anything useful. Warmup gently scales the weights into a stable region.

## 19. Day Summary
- Schedulers adjust `lr` dynamically.
- `StepLR`: Drops LR by a factor at fixed epoch intervals.
- `ReduceLROnPlateau`: Drops LR when the validation loss stalls (Requires `scheduler.step(val_loss)`).
- `OneCycleLR`: Steps every *batch* rather than every epoch, ramping up then down.